# Pseudoexperiment Study of Particle-Decay Parameter Inference

**Saqib Umair**  
Course project in statistical data analysis for particle physics

## Aim

This notebook studies how background contamination affects parameter estimation in a simulated oscillating particle-decay model. The parameters of interest are the lifetime $\tau$, oscillation frequency $\Delta m$, and modulation parameter $V$.

The analysis has two parts:

1. **Pure-signal baseline:** generate pseudoexperiments from the signal model and study the fitted parameter distributions.
2. **Background-contamination study:** inject 1%, 10%, and 20% background and compare a mismatched signal-only likelihood with a signal-plus-background likelihood.

## Model

$$p_{\rm sig}(t\mid\tau,\Delta m,V) \propto [1+V\sin(\Delta m t)]e^{-t/\tau}$$

with nominal values $\tau=1.5\times10^{-6}$ s, $\Delta m=20\times10^6$ s$^{-1}$, and $V=0.1$ over $t\in[0.5,10]\times10^{-6}$ s.

The background model is a truncated Gaussian in decay time with $\sigma=0.5\times10^{-6}$ s.

Both PDFs are normalised numerically over the accepted time range. Events are generated with accept-reject sampling and fitted with an unbinned negative log-likelihood using `iminuit`.


In [ ]:
#Importing the required libraries 
import  numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from scipy.integrate import quad
from iminuit import Minuit
import time

## 1. Pure-signal baseline

A signal-only sample is generated and compared with the nominal probability density. Repeated pseudoexperiments are then fitted to quantify the spread and bias of the fitted parameters.


In [ ]:
# Given Data 

tau= 1.5e-6 #Lifetime parameter in seconds
delta_m= 20.0e6 # Oscillation parameter in s^-1
V=0.1 # Matter/antimatter asymmetry parameter

# Time recording limits of Detector 
t_min= 0.5e-6
t_max=10.0e-6
#-unnormalised PDF
def unnormalised_pdf(t,tau,delta_m,V):
    return (1+ V*np.sin(delta_m*t))*np.exp(-t/tau)
#normalising constant 
integral, _ = quad(lambda x: unnormalised_pdf(x, tau, delta_m, V), t_min, t_max)
# Normalised Function
def normalised_pdf(t,tau,delta_m,V):
    return unnormalised_pdf(t,tau,delta_m,V)/integral
# the maximum value of the NORMALIZED PDF (f_max_normalized)
# Create the array of t-values
t_values = np.linspace(t_min, t_max, 2000)
# Calculate the NORMALIZED PDF's value at each point
pdf_values= normalised_pdf(t_values, tau, delta_m, V)
# 3. Find the highest value and add a small safety margin
f_max_normalized = np.max(pdf_values)
# Events generation 
accepted_events= []
n_events= 10000
while len(accepted_events)< n_events:
    #random points on time axis (width)
    t_random= np.random.uniform(t_min,t_max)
    #random points on the height axis
    y_random= np.random.uniform(0,f_max_normalized)
    #PDF's true height at our random time
    pdf_height= normalised_pdf(t_random,tau,delta_m,V)
    #Accept and reject
    if y_random <= pdf_height:
        accepted_events.append(t_random)
     #else: print("Reject")
#Plot the histogram
plt.figure(figsize=(10, 6))
# 1. Plot the histogram WITH density=True
plt.hist(accepted_events,bins=300,range=(t_min, t_max),density=True,label='Generated data ')
plt.plot(t_values, pdf_values, 'r', linewidth=2, label='Fitted PDF')
plt.ylabel("Events") 
plt.xlabel("Decay Time (µs)")
plt.legend()
plt.show()

### Unbinned maximum-likelihood fit

For each pseudoexperiment, the parameters are obtained by minimising

$$-\log\mathcal{L}=-\sum_i \log p(t_i\mid\tau,\Delta m,V).$$

The spread of fitted values estimates statistical precision. Bias is defined as the ensemble mean minus the generated value.


In [ ]:
def nll(tau,delta_m,V):
    if tau <= 0 or abs(V) > 1:
        return 1e12
    integral, _ = quad(lambda x: unnormalised_pdf(x, tau, delta_m, V), t_min, t_max,epsabs=1e-9, epsrel=1e-9, limit=400)
    p_values = unnormalised_pdf(accepted_events, tau, delta_m, V) / integral
    # Replace non-positive values with a tiny positive number
    p_values = np.clip(p_values, 1e-300, None)
    nll_value = -np.sum(np.log(p_values))
    return nll_value

#PSEUDOEXPERIMENT LOOP
fit_results_tau = []
fit_results_delta_m = []
fit_results_V = []


for i in range(500):
    # Generate new 10,000 event dataset
    accepted_events_list = []
    while len(accepted_events_list) < 500:
        t_random = np.random.uniform(low=t_min, high=t_max)
        y_random = np.random.uniform(low=0, high=f_max_normalized)
        
        pdf_height = normalised_pdf(t_random, tau, delta_m, V) 
        
        if y_random <= pdf_height:
            accepted_events_list.append(t_random)
    accepted_events = np.array(accepted_events_list)
    m = Minuit(nll,tau=1.5e-6,delta_m=20.0e6,V=0.1)
    m.migrad()
    fit_results_tau.append(m.values['tau'])
    fit_results_delta_m.append(m.values['delta_m'])
    fit_results_V.append(m.values['V'])
#Calculating Precision (Statistical Uncertainty)(Standard Deviation)
precision_tau = np.std(fit_results_tau)
precision_delta_m = np.std(fit_results_delta_m)
precision_V = np.std(fit_results_V)

# Calculating the Bias
mean_tau = np.mean(fit_results_tau)
mean_delta_m = np.mean(fit_results_delta_m)
mean_V = np.mean(fit_results_V)

bias_tau = mean_tau - tau
bias_delta_m = mean_delta_m - delta_m
bias_V = mean_V - V

# Calculating the Precision on the Bias 
precision_on_bias_tau = precision_tau / np.sqrt(500)
precision_on_bias_delta_m = precision_delta_m / np.sqrt(500)
precision_on_bias_V = precision_V / np.sqrt(500)
#Create a Pandas DataFrame to summarise results
summary = pd.DataFrame({
    "Parameter": ["τ (s)", "Δm (s⁻¹)", "V"],
    "True Value": [tau, delta_m, V],
    "Mean Value": [mean_tau, mean_delta_m, mean_V],
    "Bias": [bias_tau, bias_delta_m, bias_V],
    "Precision (σ)": [precision_tau, precision_delta_m, precision_V],
    "Err_Bias (σ/√N)": [precision_on_bias_tau, precision_on_bias_delta_m, precision_on_bias_V]})
print(summary.to_string(index=False))

#Plot the distributions of the fit results

# Plot for Tau (τ)
plt.figure(figsize=(10, 6))
plt.hist(fit_results_tau, bins=100, alpha=0.7, label='Fit Results', density=True, edgecolor='black')
plt.axvline(mean_tau, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_tau:.4e}')
plt.axvline(tau, color='blue', linestyle='-', linewidth=2, label=f'True Value: {tau:.4e}')
plt.title('Distribution of Fitted τ Values')
plt.xlabel('τ (s)')
plt.ylabel('Density')
plt.legend()
plt.grid(True)
plt.show()

# Plot for Delta m (Δm)
plt.figure(figsize=(10, 6))
plt.hist(fit_results_delta_m, bins=50, alpha=0.7, label='Fit Results', density=True, edgecolor='black')
plt.axvline(mean_delta_m, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_delta_m:.4f}')
plt.axvline(delta_m, color='blue', linestyle='-', linewidth=2, label=f'True Value: {delta_m:.4f}')
plt.title('Distribution of Fitted Δm Values')
plt.xlabel(r'$\Delta m\ (\mathrm{s}^{-1})$')
plt.ylabel('Density')
plt.legend()
plt.grid(True)
plt.show()


# Plot for V
plt.figure(figsize=(10, 6))
plt.hist(fit_results_V, bins=100, alpha=0.7, label='Fit Results', density=True, edgecolor='black')
plt.axvline(mean_V, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_V:.4f}')
plt.axvline(V, color='blue', linestyle='-', linewidth=2, label=f'True Value: {V:.4f}')
plt.title('Distribution of Fitted V Values')
plt.xlabel('V (arbitrary units)')
plt.ylabel('Density')
plt.legend()
plt.grid(True)
plt.show()

    

### Baseline result

In the saved ensemble, the fitted $\tau$ and $\Delta m$ means are close to their generated values. The fitted $V$ distribution shows a visible positive offset, so the baseline is not treated as a perfect closure test.


## 2. Background-contamination study

The second stage adds a background component with $\sigma=0.5\times10^{-6}$ s. Three contamination levels are considered: 1%, 10%, and 20%.

Two fit strategies are compared:

- **Signal-only fit:** ignores the injected background.
- **Signal + background fit:** includes the background fraction as a free parameter.


In [ ]:

# Constants
sigma_bkg = 0.5e-6    # Background width

n_events = 10000      # Events per pseudoexperiment
n_pseudo = 100       # Number of experiments

# Signal PDF Functions 
def unnormalised_pdf(t, tau, delta_m, V):
    return (1.0 + V * np.sin(delta_m * t)) * np.exp(-t / tau)
#I make function for handling the error and limit of quid 
def get_normalisation_constant(tau, delta_m, V):
    integral, _ = quad(lambda x: unnormalised_pdf(x, tau, delta_m, V), t_min, t_max)
    return integral

def normalised_pdf(t, tau, delta_m, V, norm=None):
    if norm is None:
        norm = get_normalisation_constant(tau, delta_m, V)
    return unnormalised_pdf(t, tau, delta_m, V) / norm
# Background PDF Functions
def unnormalised_background(t, sigma):
    return np.exp(-t**2 / (2 * sigma**2))
#I make function for handling the error and limit of quid 
def get_background_normalisation(sigma):
    integral, _ = quad(lambda x: unnormalised_background(x, sigma), t_min, t_max)
    return integral

background_norm = get_background_normalisation(sigma_bkg)

def normalised_background_pdf(t):
    return unnormalised_background(t, sigma_bkg) / background_norm

#  Event Generator Functions
# Signal Events only
def generate_signal_events(n_events, tau, delta_m, V):
    t_values = np.linspace(t_min, t_max, 1000)
    norm = get_normalisation_constant(tau, delta_m, V)
    pdf_values = normalised_pdf(t_values, tau, delta_m, V, norm=norm)
    f_max = np.max(pdf_values) * 1.1

    accepted = []
    while len(accepted) < n_events:
        t_rand = np.random.uniform(t_min, t_max)
        y_rand = np.random.uniform(0, f_max)
        pdf_val = normalised_pdf(t_rand, tau, delta_m, V, norm=norm)
        if y_rand <= pdf_val:
            accepted.append(t_rand)
    return np.array(accepted)
#Background Events only
def generate_background_events(n_events):
    f_max = normalised_background_pdf(t_min) * 1.1
    accepted = []
    while len(accepted) < n_events:
        t_rand = np.random.uniform(t_min, t_max)
        y_rand = np.random.uniform(0, f_max)
        pdf_val = normalised_background_pdf(t_rand)
        if y_rand <= pdf_val:
            accepted.append(t_rand)
    return np.array(accepted)
#Combine events
def generate_combined_events(n_total, f_bkg):
    n_bkg = int(n_total * f_bkg)
    n_sig = n_total - n_bkg
    sig_events = generate_signal_events(n_sig, tau, delta_m, V)
    bkg_events = generate_background_events(n_bkg)
    combined = np.concatenate([sig_events, bkg_events])
    np.random.shuffle(combined)
    return combined



### 2.1 Mismatched likelihood: background ignored

The contaminated samples are first fitted with the signal-only PDF to measure how model misspecification propagates into the inferred physics parameters.


In [ ]:
# For case A
#  NLL Fit Function Classes
# for Encapsulating of Data, I make class this time for NLL
class NLLSignalOnly:
    def __init__(self, data):
        self.data = data
    def __call__(self, tau, delta_m, V):
        try:
            norm = get_normalisation_constant(tau, delta_m, V)
            if norm <= 0: return 1e9
        except Exception:
            return 1e9
        p_vals = unnormalised_pdf(self.data, tau, delta_m, V) / norm
        if np.any(p_vals <= 0): return 1e9
        return -np.sum(np.log(p_vals))

### 2.2 Signal-plus-background likelihood

The same samples are then fitted with

$$p_{\rm total}(t)=(1-f_{\rm bkg})p_{\rm sig}(t)+f_{\rm bkg}p_{\rm bkg}(t),$$

where $\tau$, $\Delta m$, $V$, and $f_{\rm bkg}$ are free parameters while the background width is fixed.


In [ ]:
class NLLSignalPlusBackground:
    def __init__(self, data):
        self.data = data
        self.p_bkg = normalised_background_pdf(self.data)
    def __call__(self, tau, delta_m, V, f_bkg):
        try:
            norm = get_normalisation_constant(tau, delta_m, V)
            if norm <= 0: return 1e9
        except Exception:
            return 1e9
        p_sig = unnormalised_pdf(self.data, tau, delta_m, V) / norm
        p_total = (1 - f_bkg) * p_sig + f_bkg * self.p_bkg
        if np.any(p_total <= 0): return 1e9
        return -np.sum(np.log(p_total))


In [ ]:
# Main Pseudoexperiment Loop
background_levels = [0.01, 0.10, 0.20]
results_wrong = {}
results_correct = {}
for f in background_levels:
    results_wrong[f] = {'V': [], 'tau': [], 'delta_m': []}
    results_correct[f] = {'V': [], 'tau': [], 'delta_m': [], 'f_bkg': []}

limits_tau = (1.0e-6, 2.0e-6)
limits_delta_m = (15.0e6, 25.0e6)
limits_V = (-0.5, 0.5)
limits_f_bkg = (0.0, 0.5)

start_time = time.time()
for f_true in background_levels:
    for i in range(n_pseudo):
        data = generate_combined_events(n_events, f_true)

        # Wrong Fit
        nll_wrong = NLLSignalOnly(data)
        fit_wrong = Minuit(nll_wrong, tau=tau, delta_m=delta_m, V=V)
        fit_wrong.limits["tau"] = limits_tau
        fit_wrong.limits["delta_m"] = limits_delta_m
        fit_wrong.limits["V"] = limits_V
        fit_wrong.migrad()
        if fit_wrong.valid:
            results_wrong[f_true]['V'].append(fit_wrong.values['V'])
            results_wrong[f_true]['tau'].append(fit_wrong.values['tau'])
            results_wrong[f_true]['delta_m'].append(fit_wrong.values['delta_m'])

        # Correct Fit
        nll_correct = NLLSignalPlusBackground(data)
        fit_correct = Minuit(nll_correct, tau=tau, delta_m=delta_m, V=V, f_bkg=f_true)
        fit_correct.limits["tau"] = limits_tau
        fit_correct.limits["delta_m"] = limits_delta_m
        fit_correct.limits["V"] = limits_V
        fit_correct.limits["f_bkg"] = limits_f_bkg
        fit_correct.migrad()
        if fit_correct.valid:
            results_correct[f_true]['V'].append(fit_correct.values['V'])
            results_correct[f_true]['tau'].append(fit_correct.values['tau'])
            results_correct[f_true]['delta_m'].append(fit_correct.values['delta_m'])
            results_correct[f_true]['f_bkg'].append(fit_correct.values['f_bkg'])


In [ ]:
#  Plotting the data
# Nominal values for comparison
nominal_values = {'tau': tau, 'delta_m': delta_m, 'V': V}
# Parameters to plot and background fractions to compare
parameters_to_plot = ['tau', 'delta_m', 'V']
fractions = [0.01, 0.10, 0.20]
colors = {0.01: 'green', 0.10: 'orange', 0.20: 'blue'}
param_labels = {
    'tau': r'$\tau$ (s)', 
    'delta_m': r'$\Delta m$ (s⁻¹)', 
    'V': 'V (unitless)'}
param_titles = {
            'tau': r'$\tau$ Comparison: Signal Only Fit vs Sig+Bkg Fit',
            'delta_m': r'$\Delta m$ Comparison: Signal Only Fit vs Sig+Bkg Fit',
            'V': 'V Comparison: Signal Only Fit vs Sig+Bkg Fit'}
# Loop through each parameter (τ, delta_m, V)
for param_key in parameters_to_plot:
    # Create a figure with 3 subplots (one for each background fraction)
    fig, axes = plt.subplots(1, 3, figsize=(24, 7))
    fig.suptitle(param_titles[param_key], fontsize=16, fontweight='bold', y=1.02) 
# Loop through each background fraction (1%, 10%, 20
    for j, frac in enumerate(fractions):
        ax = axes[j]
        frac_label = f"{frac*100:.0f}% Background"
        color = colors[frac]
         # Extracting fitted results
        data_A = results_wrong[frac][param_key] 
        data_B = results_correct[frac][param_key] 
        
        mean_A = np.mean(data_A)
        mean_B = np.mean(data_B)
        
        true_val = nominal_values[param_key] 
        # Combine data for bin range and create histogram bins
        all_data = np.concatenate([data_A, data_B])
        bins = np.linspace(np.min(all_data), np.max(all_data), 40) 
        # Plotting histograms for both fits
        ax.hist(data_B, bins=bins, color='grey', alpha=0.7, 
                label='Sig+Bkg Fit', edgecolor='black') 
        ax.hist(data_A, bins=bins, color=color, alpha=0.6, 
                label='Signal Only Fit', edgecolor='black') 
        # vertical lines for mean values and true value
        ax.axvline(mean_B, color='black', linestyle='--', linewidth=2, label='Sig+Bkg Mean') 
        ax.axvline(mean_A, color=color, linestyle='--', linewidth=2, label='Signal Only Mean') 
        
        ax.axvline(true_val, color='blue', linestyle='-', linewidth=2, label='True Value') 
        # Set titles, labels, and grid
        ax.set_title(f"Comparison at {frac_label}", fontsize=14) 
        ax.set_xlabel(param_labels[param_key], fontsize=12)
        ax.set_ylabel("Count", fontsize=12)
        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.5)
        #scientific notation for τ and Δm axes
        if param_key in ['tau', 'delta_m']:
            ax.ticklabel_format(style='sci', axis='x', scilimits=(0,0))

    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) 
    plt.show()



In [ ]:
# Summary
def print_text_summary(results_wrong, results_correct, nom_values):
    print(f"{'Background%':<12} {'FitType':<12} {'Parameter':<10} {'True':<12} {'Mean':<12} {'Bias':<12} {'Precision':<12}")
    print("-" * 74)

    fractions = [0.01, 0.10, 0.20]
    params = ['tau', 'delta_m', 'V']
    param_names = {'tau': 'Tau', 'delta_m': 'delta_m', 'V': 'V'}

    for f_true in fractions:
        bg_label = int(f_true * 100)
        
        for param in params:
            param_label = param_names[param]
            true_val = nom_values[param]

            # Signal Only Fit
            res_A = np.array(results_wrong[f_true][param])
            mean_A = np.mean(res_A)
            bias_A = mean_A - true_val
            prec_A = np.std(res_A)
            
            print(f"{bg_label:<12} {'Signal':<12} {param_label:<10} {true_val:<12.2e} {mean_A:<12.2e} {bias_A:<12.2e} {prec_A:<12.2e}")

            # Sig+Bkg Fit
            res_B = np.array(results_correct[f_true][param])
            mean_B = np.mean(res_B)
            bias_B = mean_B - true_val
            prec_B = np.std(res_B)
            
            print(f"{bg_label:<12} {'Background':<12} {param_label:<10} {true_val:<12.2e} {mean_B:<12.2e} {bias_B:<12.2e} {prec_B:<12.2e}")

# Call the summary function
print_text_summary(results_wrong, results_correct, nominal_values)

## 3. Interpretation

The saved ensemble shows a clear failure mode when background is ignored. At 20% contamination, the signal-only fit shifts the lifetime from the generated $1.50\times10^{-6}$ s to about $1.24\times10^{-6}$ s, while $V$ moves from 0.10 to about 0.055.

The signal-plus-background fit recovers $\tau$ close to $1.50\times10^{-6}$ s and $V$ close to 0.10 in the same ensemble.

The main result is that **likelihood misspecification can create a systematic parameter bias much larger than the statistical spread**. Explicitly modelling the nuisance component largely removes that bias, although the additional free parameter can reduce statistical precision through correlations.
